# Loading data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from itertools import combinations


required_files = [
    "raw_QOS_OFA_fresh_30seeds.xlsx",
    "raw_IT_QOS_OFA_fresh_30seeds.xlsx",
    "raw_IT_SIA_QOS_OFA_fresh_30seeds.xlsx",
]

candidate_dirs = [
    Path.cwd() / "analysis_exports",
    Path.cwd().parent / "analysis_exports",
    Path.cwd().parent.parent / "analysis_exports",
]

DATA_DIR = None

for candidate in candidate_dirs:
    if candidate.exists() and all((candidate / f).exists() for f in required_files):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    print("Could not automatically find all required files.")
    print("\nCurrent working directory:")
    print(Path.cwd())

    print("\nSearched these folders:")
    for candidate in candidate_dirs:
        print(candidate)

    print("\nFiles found recursively from current directory:")
    for file_name in required_files:
        matches = list(Path.cwd().rglob(file_name))
        print(f"\n{file_name}:")
        if matches:
            for m in matches:
                print("  ", m)
        else:
            print("   NOT FOUND")

    raise FileNotFoundError(
        "Could not find all raw Excel files. Set DATA_DIR manually to the folder containing them."
    )

print("Using DATA_DIR:")
print(DATA_DIR)


RAW_FILES = {
    "QOS-OFA": DATA_DIR / "raw_QOS_OFA_fresh_30seeds.xlsx",
    "IT-QOS-OFA": DATA_DIR / "raw_IT_QOS_OFA_fresh_30seeds.xlsx",
    "IT-SIA-QOS-OFA": DATA_DIR / "raw_IT_SIA_QOS_OFA_fresh_30seeds.xlsx",
}

for alg, path in RAW_FILES.items():
    print(f"{alg}: {path} | exists = {path.exists()}")

RAW_SHEET = "raw_per_run"

EXPECTED_FUNCTIONS = [3, 5, 6, 9, 11, 16, 20, 22, 24, 27, 28]

METRIC = "cec_error"
LOWER_IS_BETTER = True
ALPHA = 0.05

Using DATA_DIR:
c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports
QOS-OFA: c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\raw_QOS_OFA_fresh_30seeds.xlsx | exists = True
IT-QOS-OFA: c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\raw_IT_QOS_OFA_fresh_30seeds.xlsx | exists = True
IT-SIA-QOS-OFA: c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\raw_IT_SIA_QOS_OFA_fresh_30seeds.xlsx | exists = True


In [7]:
def load_raw_algorithm_file(path, algorithm_name):
    """
    Loads one algorithm workbook and keeps only the columns needed
    for statistical analysis.
    """

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_excel(path, sheet_name=RAW_SHEET)

    required_cols = [
        "cec_func_num",
        "objective_name",
        "seed",
        "final_best",
        "cec_error",
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{path.name} is missing required columns: {missing_cols}"
        )

    clean_df = df[required_cols].copy()
    clean_df["algorithm"] = algorithm_name

    clean_df["cec_func_num"] = clean_df["cec_func_num"].astype(int)
    clean_df["seed"] = clean_df["seed"].astype(int)
    clean_df["final_best"] = pd.to_numeric(clean_df["final_best"], errors="coerce")
    clean_df["cec_error"] = pd.to_numeric(clean_df["cec_error"], errors="coerce")

    clean_df = clean_df[
        [
            "algorithm",
            "cec_func_num",
            "objective_name",
            "seed",
            "final_best",
            "cec_error",
        ]
    ]

    return clean_df


raw_dfs = []

for algorithm_name, path in RAW_FILES.items():
    temp_df = load_raw_algorithm_file(path, algorithm_name)
    raw_dfs.append(temp_df)

raw_all_df = pd.concat(raw_dfs, ignore_index=True)

raw_all_df = raw_all_df.sort_values(
    ["cec_func_num", "seed", "algorithm"]
).reset_index(drop=True)

print("Combined raw data:")
display(raw_all_df.head(20))

print("\nRows per algorithm:")
display(raw_all_df["algorithm"].value_counts())

print("\nRows per algorithm/function:")
display(
    raw_all_df
    .groupby(["algorithm", "cec_func_num"])
    .size()
    .reset_index(name="n_runs")
)

Combined raw data:


,algorithm,cec_func_num,objective_name,seed,final_best,cec_error
0,IT-QOS-OFA,3,CEC Zakharov,1,300.824251,0.824251
1,IT-SIA-QOS-OFA,3,CEC Zakharov,1,300.106903,0.106903
2,QOS-OFA,3,CEC Zakharov,1,300.173950,0.173950
3,IT-QOS-OFA,3,CEC Zakharov,2,302.449178,2.449178
4,IT-SIA-QOS-OFA,3,CEC Zakharov,2,300.006561,0.006561
5,QOS-OFA,3,CEC Zakharov,2,300.178101,0.178101
6,IT-QOS-OFA,3,CEC Zakharov,3,300.811962,0.811962
7,IT-SIA-QOS-OFA,3,CEC Zakharov,3,300.020294,0.020294
8,QOS-OFA,3,CEC Zakharov,3,300.168640,0.168640
9,IT-QOS-OFA,3,CEC Zakharov,4,301.046184,1.046184



Rows per algorithm:


algorithm
IT-QOS-OFA        330
IT-SIA-QOS-OFA    330
QOS-OFA           330
Name: count, dtype: int64


Rows per algorithm/function:


,algorithm,cec_func_num,n_runs
0,IT-QOS-OFA,3,30
1,IT-QOS-OFA,5,30
2,IT-QOS-OFA,6,30
3,IT-QOS-OFA,9,30
4,IT-QOS-OFA,11,30
5,IT-QOS-OFA,16,30
6,IT-QOS-OFA,20,30
7,IT-QOS-OFA,22,30
8,IT-QOS-OFA,24,30
9,IT-QOS-OFA,27,30


# Friedman Test

In [ ]:
friedman_input = (
    raw_all_df
    .groupby(["cec_func_num", "objective_name", "algorithm"], as_index=False)
    .agg(
        error_median=("cec_error", "median"),
        error_mean=("cec_error", "mean"),
        error_std=("cec_error", "std"),
        error_best=("cec_error", "min"),
        error_worst=("cec_error", "max"),
    )
)

friedman_wide = (
    friedman_input
    .pivot_table(
        index=["cec_func_num", "objective_name"],
        columns="algorithm",
        values="error_median",
        aggfunc="first"
    )
    .reset_index()
)

friedman_wide = friedman_wide[
    [
        "cec_func_num",
        "objective_name",
        "QOS-OFA",
        "IT-QOS-OFA",
        "IT-SIA-QOS-OFA",
    ]
]

print("Friedman input table using median CEC error:")
display(friedman_wide)


algorithm_cols = ["QOS-OFA", "IT-QOS-OFA", "IT-SIA-QOS-OFA"]

data = friedman_wide[algorithm_cols].to_numpy(dtype=float)

# Lower error receives better rank.
ranks = np.array([rankdata(row, method="average") for row in data])

rank_df = friedman_wide[["cec_func_num", "objective_name"]].copy()

for i, alg in enumerate(algorithm_cols):
    rank_df[f"{alg}_rank"] = ranks[:, i]

average_ranks = pd.DataFrame({
    "algorithm": algorithm_cols,
    "average_rank": ranks.mean(axis=0),
}).sort_values("average_rank").reset_index(drop=True)

print("\nPer-function ranks:")
display(rank_df)

print("\nAverage ranks:")
display(average_ranks)

Friedman input table using median CEC error:


algorithm,cec_func_num,objective_name,QOS-OFA,IT-QOS-OFA,IT-SIA-QOS-OFA
0,3,CEC Zakharov,0.283447,1.196119,0.104767
1,5,CEC Rastrigin,48.757935,104.831609,130.269928
2,6,CEC Expanded Scaffer's F6,13.134979,13.766882,11.788757
3,9,CEC Levy,0.099609,2.006804,26.492279
4,11,CEC Hybrid Function 1,22.670593,66.084845,59.787476
5,16,CEC Hybrid Function 6,263.244873,621.188127,975.546631
6,20,CEC Hybrid Function 10,185.239868,227.994241,271.064209
7,22,CEC Composition Function 2,100.508911,116.328370,100.002319
8,24,CEC Composition Function 4,412.235107,444.867985,503.159790
9,27,CEC Composition Function 7,455.984131,464.944285,473.654663



Per-function ranks:


algorithm,cec_func_num,objective_name,QOS-OFA_rank,IT-QOS-OFA_rank,IT-SIA-QOS-OFA_rank
0,3,CEC Zakharov,2.0,3.0,1.0
1,5,CEC Rastrigin,1.0,2.0,3.0
2,6,CEC Expanded Scaffer's F6,2.0,3.0,1.0
3,9,CEC Levy,1.0,2.0,3.0
4,11,CEC Hybrid Function 1,1.0,3.0,2.0
5,16,CEC Hybrid Function 6,1.0,2.0,3.0
6,20,CEC Hybrid Function 10,1.0,2.0,3.0
7,22,CEC Composition Function 2,2.0,3.0,1.0
8,24,CEC Composition Function 4,1.0,2.0,3.0
9,27,CEC Composition Function 7,1.0,2.0,3.0



Average ranks:


,algorithm,average_rank
0,QOS-OFA,1.363636
1,IT-SIA-QOS-OFA,2.181818
2,IT-QOS-OFA,2.454545


In [10]:
friedman_stat, friedman_p = friedmanchisquare(
    friedman_wide["QOS-OFA"],
    friedman_wide["IT-QOS-OFA"],
    friedman_wide["IT-SIA-QOS-OFA"],
)

friedman_summary = pd.DataFrame({
    "test": ["Friedman"],
    "metric": ["Median CEC error per function"],
    "n_functions": [len(friedman_wide)],
    "n_algorithms": [len(algorithm_cols)],
    "statistic": [friedman_stat],
    "p_value": [friedman_p],
    "significant_0_05": [friedman_p < 0.05],
})

display(friedman_summary)

,test,metric,n_functions,n_algorithms,statistic,p_value,significant_0_05
0,Friedman,Median CEC error per function,11,3,7.090909,0.028856,True


# Wilcoxon signed rank test

In [11]:
def format_p_value(p):
    if p < 1e-4:
        return f"{p:.2E}"
    return f"{p:.4f}"


def get_paired_values(df, func_num, alg_a, alg_b, metric="cec_error"):
    a = (
        df[
            (df["cec_func_num"] == func_num) &
            (df["algorithm"] == alg_a)
        ]
        .sort_values("seed")
        [["seed", metric]]
        .rename(columns={metric: f"{alg_a}_value"})
    )

    b = (
        df[
            (df["cec_func_num"] == func_num) &
            (df["algorithm"] == alg_b)
        ]
        .sort_values("seed")
        [["seed", metric]]
        .rename(columns={metric: f"{alg_b}_value"})
    )

    paired = pd.merge(a, b, on="seed", how="inner")

    if len(paired) != 30:
        raise ValueError(
            f"Expected 30 paired runs for CEC {func_num}, "
            f"{alg_a} vs {alg_b}, but found {len(paired)}."
        )

    return paired


def wilcoxon_one_function(df, func_num, alg_a, alg_b, metric="cec_error"):
    paired = get_paired_values(df, func_num, alg_a, alg_b, metric)

    values_a = paired[f"{alg_a}_value"].to_numpy(dtype=float)
    values_b = paired[f"{alg_b}_value"].to_numpy(dtype=float)

    differences = values_a - values_b

    if np.allclose(differences, 0):
        statistic = 0.0
        p_value = 1.0
    else:
        statistic, p_value = wilcoxon(
            values_a,
            values_b,
            zero_method="wilcox",
            alternative="two-sided",
            mode="auto"
        )

    median_a = np.median(values_a)
    median_b = np.median(values_b)

    mean_a = np.mean(values_a)
    mean_b = np.mean(values_b)

    # Lower error is better.
    if p_value < ALPHA:
        if median_a < median_b:
            sign = "+"
        elif median_a > median_b:
            sign = "-"
        else:
            sign = "="
    else:
        sign = "="

    return {
        "statistic": statistic,
        "p_value": p_value,
        "p_value_sign": f"{format_p_value(p_value)}({sign})",
        "sign": sign,
        "median_A": median_a,
        "median_B": median_b,
        "mean_A": mean_a,
        "mean_B": mean_b,
        "n_pairs": len(paired),
    }


pairwise_comparisons = [
    ("QOS-OFA", "IT-QOS-OFA"),
    ("QOS-OFA", "IT-SIA-QOS-OFA"),
    ("IT-QOS-OFA", "IT-SIA-QOS-OFA"),
]

objective_names = (
    raw_all_df
    .drop_duplicates("cec_func_num")
    .set_index("cec_func_num")["objective_name"]
    .to_dict()
)

wilcoxon_rows = []

for func_num in EXPECTED_FUNCTIONS:
    row = {
        "cec_func_num": func_num,
        "objective_name": objective_names.get(func_num, f"CEC {func_num}"),
    }

    for alg_a, alg_b in pairwise_comparisons:
        result = wilcoxon_one_function(
            raw_all_df,
            func_num,
            alg_a,
            alg_b,
            metric=METRIC
        )

        comparison_name = f"{alg_a} vs {alg_b}"

        row[f"{comparison_name} p(sign)"] = result["p_value_sign"]
        row[f"{comparison_name} raw_p"] = result["p_value"]
        row[f"{comparison_name} sign"] = result["sign"]
        row[f"{comparison_name} median_{alg_a}"] = result["median_A"]
        row[f"{comparison_name} median_{alg_b}"] = result["median_B"]
        row[f"{comparison_name} n_pairs"] = result["n_pairs"]

    wilcoxon_rows.append(row)

wilcoxon_per_function = pd.DataFrame(wilcoxon_rows)

display(wilcoxon_per_function)

,cec_func_num,objective_name,QOS-OFA vs IT-QOS-OFA p(sign),QOS-OFA vs IT-QOS-OFA raw_p,QOS-OFA vs IT-QOS-OFA sign,QOS-OFA vs IT-QOS-OFA median_QOS-OFA,QOS-OFA vs IT-QOS-OFA median_IT-QOS-OFA,QOS-OFA vs IT-QOS-OFA n_pairs,QOS-OFA vs IT-SIA-QOS-OFA p(sign),QOS-OFA vs IT-SIA-QOS-OFA raw_p,QOS-OFA vs IT-SIA-QOS-OFA sign,QOS-OFA vs IT-SIA-QOS-OFA median_QOS-OFA,QOS-OFA vs IT-SIA-QOS-OFA median_IT-SIA-QOS-OFA,QOS-OFA vs IT-SIA-QOS-OFA n_pairs,IT-QOS-OFA vs IT-SIA-QOS-OFA p(sign),IT-QOS-OFA vs IT-SIA-QOS-OFA raw_p,IT-QOS-OFA vs IT-SIA-QOS-OFA sign,IT-QOS-OFA vs IT-SIA-QOS-OFA median_IT-QOS-OFA,IT-QOS-OFA vs IT-SIA-QOS-OFA median_IT-SIA-QOS-OFA,IT-QOS-OFA vs IT-SIA-QOS-OFA n_pairs
0,3,CEC Zakharov,6.29E-05(+),6.286614e-05,+,0.283447,1.196119,30,0.0549(=),5.492163e-02,=,0.283447,0.104767,30,7.91E-05(-),7.910654e-05,-,1.196119,0.104767,30
1,5,CEC Rastrigin,3.54E-08(+),3.539026e-08,+,48.757935,104.831609,30,1.86E-09(+),1.862645e-09,+,48.757935,130.269928,30,0.0047(+),4.664805e-03,+,104.831609,130.269928,30
2,6,CEC Expanded Scaffer's F6,9.31E-09(+),9.313226e-09,+,13.134979,13.766882,30,1.86E-09(-),1.862645e-09,-,13.134979,11.788757,30,1.86E-09(-),1.862645e-09,-,13.766882,11.788757,30
3,9,CEC Levy,1.86E-09(+),1.862645e-09,+,0.099609,2.006804,30,1.86E-09(+),1.862645e-09,+,0.099609,26.492279,30,4.71E-07(+),4.712492e-07,+,2.006804,26.492279,30
4,11,CEC Hybrid Function 1,1.86E-09(+),1.862645e-09,+,22.670593,66.084845,30,3.73E-09(+),3.725290e-09,+,22.670593,59.787476,30,0.5978(=),5.978078e-01,=,66.084845,59.787476,30
5,16,CEC Hybrid Function 6,4.71E-07(+),4.712492e-07,+,263.244873,621.188127,30,3.73E-09(+),3.725290e-09,+,263.244873,975.546631,30,0.0001(+),1.233369e-04,+,621.188127,975.546631,30
6,20,CEC Hybrid Function 10,0.0028(+),2.766320e-03,+,185.239868,227.994241,30,1.86E-08(+),1.862645e-08,+,185.239868,271.064209,30,0.0054(+),5.382780e-03,+,227.994241,271.064209,30
7,22,CEC Composition Function 2,1.86E-09(+),1.862645e-09,+,100.508911,116.328370,30,0.0007(-),7.296018e-04,-,100.508911,100.002319,30,1.86E-09(-),1.862645e-09,-,116.328370,100.002319,30
8,24,CEC Composition Function 4,1.30E-08(+),1.303852e-08,+,412.235107,444.867985,30,9.31E-09(+),9.313226e-09,+,412.235107,503.159790,30,3.24E-06(+),3.239140e-06,+,444.867985,503.159790,30
9,27,CEC Composition Function 7,0.0002(+),2.091266e-04,+,455.984131,464.944285,30,1.86E-08(+),1.862645e-08,+,455.984131,473.654663,30,9.90E-05(+),9.902567e-05,+,464.944285,473.654663,30


In [12]:
sign_summary_rows = []

for alg_a, alg_b in pairwise_comparisons:
    comparison_name = f"{alg_a} vs {alg_b}"
    sign_col = f"{comparison_name} sign"

    counts = wilcoxon_per_function[sign_col].value_counts()

    sign_summary_rows.append({
        "comparison": comparison_name,
        "+ first algorithm significantly better": counts.get("+", 0),
        "= no significant difference": counts.get("=", 0),
        "- first algorithm significantly worse": counts.get("-", 0),
    })

wilcoxon_sign_summary = pd.DataFrame(sign_summary_rows)

display(wilcoxon_sign_summary)

,comparison,+ first algorithm significantly better,= no significant difference,- first algorithm significantly worse
0,QOS-OFA vs IT-QOS-OFA,11,0,0
1,QOS-OFA vs IT-SIA-QOS-OFA,7,2,2
2,IT-QOS-OFA vs IT-SIA-QOS-OFA,6,2,3


In [13]:
OUTPUT_PATH = DATA_DIR / "statistical_analysis_results_from_raw_runs.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    raw_all_df.to_excel(
        writer,
        sheet_name="raw_cleaned_all_runs",
        index=False
    )

    friedman_wide.to_excel(
        writer,
        sheet_name="friedman_input",
        index=False
    )

    rank_df.to_excel(
        writer,
        sheet_name="friedman_ranks",
        index=False
    )

    average_ranks.to_excel(
        writer,
        sheet_name="average_ranks",
        index=False
    )

    friedman_summary.to_excel(
        writer,
        sheet_name="friedman_summary",
        index=False
    )

    wilcoxon_per_function.to_excel(
        writer,
        sheet_name="wilcoxon_per_function",
        index=False
    )

    wilcoxon_sign_summary.to_excel(
        writer,
        sheet_name="wilcoxon_sign_summary",
        index=False
    )

print(f"Saved statistical analysis workbook to: {OUTPUT_PATH}")

Saved statistical analysis workbook to: c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\statistical_analysis_results_from_raw_runs.xlsx


# Numerical analysis

In [ ]:
BASELINE = "QOS-OFA"
PROPOSED_ALGORITHMS = ["IT-QOS-OFA", "IT-SIA-QOS-OFA"]

median_error_df = (
    raw_all_df
    .groupby(["cec_func_num", "objective_name", "algorithm"], as_index=False)
    .agg(
        median_error=("cec_error", "median"),
        mean_error=("cec_error", "mean"),
        std_error=("cec_error", "std"),
        best_error=("cec_error", "min"),
        worst_error=("cec_error", "max"),
    )
 )

observed_range_df = (
    raw_all_df
    .groupby(["cec_func_num", "objective_name"], as_index=False)
    .agg(
        observed_min_error=("cec_error", "min"),
        observed_max_error=("cec_error", "max"),
    )
)

observed_range_df["observed_error_range"] = (
    observed_range_df["observed_max_error"] -
    observed_range_df["observed_min_error"]
)

# Safety check: the observed range must be positive for normalization
zero_range_functions = observed_range_df[
    observed_range_df["observed_error_range"] == 0
]

if not zero_range_functions.empty:
    display(zero_range_functions)
    raise ValueError(
        "At least one CEC function has observed_error_range = 0. "
        "Normalization cannot be computed for that function."
    )


qos_median_df = (
    median_error_df[median_error_df["algorithm"] == BASELINE]
    [["cec_func_num", "objective_name", "median_error"]]
    .rename(columns={"median_error": "qos_median_error"})
)
-

proposed_df = median_error_df[
    median_error_df["algorithm"].isin(PROPOSED_ALGORITHMS)
].copy()

numerical_qos_relative = (
    proposed_df
    .merge(qos_median_df, on=["cec_func_num", "objective_name"], how="left")
    .merge(observed_range_df, on=["cec_func_num", "objective_name"], how="left")
)

# Raw difference
numerical_qos_relative["error_difference_vs_qos"] = (
    numerical_qos_relative["median_error"] -
    numerical_qos_relative["qos_median_error"]
)

# Scale-normalized signed difference
numerical_qos_relative["signed_normalized_difference_percent"] = (
    100.0 *
    numerical_qos_relative["error_difference_vs_qos"] /
    numerical_qos_relative["observed_error_range"]
)

# QOS-relative retained performance
numerical_qos_relative["qos_relative_retention_percent"] = (
    100.0 -
    numerical_qos_relative["signed_normalized_difference_percent"]
)

# Closeness to QOS-OFA
numerical_qos_relative["closeness_to_qos_percent"] = (
    100.0 -
    numerical_qos_relative["signed_normalized_difference_percent"].abs()
)

# Direction label
numerical_qos_relative["direction_vs_qos"] = np.select(
    [
        numerical_qos_relative["error_difference_vs_qos"] < 0,
        numerical_qos_relative["error_difference_vs_qos"] > 0,
    ],
    [
        "better_than_QOS",
        "worse_than_QOS",
    ],
    default="same_as_QOS"
)


numerical_qos_relative = numerical_qos_relative[
    [
        "cec_func_num",
        "objective_name",
        "algorithm",
        "qos_median_error",
        "median_error",
        "error_difference_vs_qos",
        "observed_min_error",
        "observed_max_error",
        "observed_error_range",
        "signed_normalized_difference_percent",
        "qos_relative_retention_percent",
        "closeness_to_qos_percent",
        "direction_vs_qos",
        "mean_error",
        "std_error",
        "best_error",
        "worst_error",
    ]
].sort_values(["algorithm", "cec_func_num"]).reset_index(drop=True)

display(numerical_qos_relative)

,cec_func_num,objective_name,algorithm,qos_median_error,median_error,error_difference_vs_qos,observed_min_error,observed_max_error,observed_error_range,signed_normalized_difference_percent,qos_relative_retention_percent,closeness_to_qos_percent,direction_vs_qos,mean_error,std_error,best_error,worst_error
0,3,CEC Zakharov,IT-QOS-OFA,0.283447,1.196119,0.912672,0.002014,15.527679,15.525665,5.878474,94.121526,94.121526,worse_than_QOS,2.436207,3.641462,0.229113,15.527679
1,5,CEC Rastrigin,IT-QOS-OFA,48.757935,104.831609,56.073674,24.906006,251.833801,226.927795,24.709919,75.290081,75.290081,worse_than_QOS,106.813636,31.325394,49.345574,163.626186
2,6,CEC Expanded Scaffer's F6,IT-QOS-OFA,13.134979,13.766882,0.631903,10.450012,14.235301,3.785289,16.693652,83.306348,83.306348,worse_than_QOS,13.763935,0.245770,13.105189,14.235301
3,9,CEC Levy,IT-QOS-OFA,0.099609,2.006804,1.907194,0.007080,453.618286,453.611206,0.420447,99.579553,99.579553,worse_than_QOS,3.946485,7.841389,0.832761,43.723286
4,11,CEC Hybrid Function 1,IT-QOS-OFA,22.670593,66.084845,43.414252,16.136353,900.794749,884.658397,4.907459,95.092541,95.092541,worse_than_QOS,143.447544,197.531833,29.154732,900.794749
5,16,CEC Hybrid Function 6,IT-QOS-OFA,263.244873,621.188127,357.943254,4.785767,1454.330566,1449.544800,24.693494,75.306506,75.306506,worse_than_QOS,620.418791,201.747811,153.213904,1089.811619
6,20,CEC Hybrid Function 10,IT-QOS-OFA,185.239868,227.994241,42.754373,62.182373,427.763916,365.581543,11.694894,88.305106,88.305106,worse_than_QOS,218.867115,68.075043,107.158966,359.774272
7,22,CEC Composition Function 2,IT-QOS-OFA,100.508911,116.328370,15.819459,100.000977,117.701567,17.700590,89.372491,10.627509,10.627509,worse_than_QOS,116.078350,0.900819,114.545842,117.701567
8,24,CEC Composition Function 4,IT-QOS-OFA,412.235107,444.867985,32.632878,374.608398,569.303223,194.694824,16.761040,83.238960,83.238960,worse_than_QOS,444.149137,19.542250,408.055394,484.924707
9,27,CEC Composition Function 7,IT-QOS-OFA,455.984131,464.944285,8.960155,450.041748,493.699219,43.657471,20.523760,79.476240,79.476240,worse_than_QOS,464.568323,7.890212,452.203492,479.417146


In [ ]:
# Summary of QOS-relative numerical performance
numerical_summary = (
    numerical_qos_relative
    .groupby("algorithm", as_index=False)
    .agg(
        mean_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "mean"
        ),
        median_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "median"
        ),
        functions_better_than_qos=(
            "direction_vs_qos", lambda x: int((x == "better_than_QOS").sum())
        ),
        functions_worse_than_qos=(
            "direction_vs_qos", lambda x: int((x == "worse_than_QOS").sum())
        ),
        functions_same_as_qos=(
            "direction_vs_qos", lambda x: int((x == "same_as_QOS").sum())
        ),
    )
)

display(numerical_summary)

,algorithm,mean_qos_relative_retention_percent,median_qos_relative_retention_percent,functions_better_than_qos,functions_worse_than_qos,functions_same_as_qos
0,IT-QOS-OFA,79.912861,83.306348,0,11,0
1,IT-SIA-QOS-OFA,84.920631,94.181654,4,7,0


In [16]:
better_only_summary = (
    numerical_qos_relative[
        numerical_qos_relative["qos_relative_retention_percent"] > 100
    ]
    .groupby("algorithm", as_index=False)
    .agg(
        n_better_functions=("cec_func_num", "count"),
        mean_improvement_when_better_percent=(
            "qos_relative_retention_percent",
            lambda x: (x - 100).mean()
        ),
        median_improvement_when_better_percent=(
            "qos_relative_retention_percent",
            lambda x: (x - 100).median()
        ),
        max_improvement_percent=(
            "qos_relative_retention_percent",
            lambda x: (x - 100).max()
        ),
    )
)

display(better_only_summary)

,algorithm,n_better_functions,mean_improvement_when_better_percent,median_improvement_when_better_percent,max_improvement_percent
0,IT-SIA-QOS-OFA,4,9.963256,2.006438,35.564576


In [17]:
NUMERICAL_OUTPUT_PATH = DATA_DIR / "numerical_qos_relative_analysis.xlsx"

with pd.ExcelWriter(NUMERICAL_OUTPUT_PATH, engine="openpyxl") as writer:
    numerical_qos_relative.to_excel(
        writer,
        sheet_name="qos_relative_by_function",
        index=False
    )

    numerical_summary.to_excel(
        writer,
        sheet_name="qos_relative_summary",
        index=False
    )

    better_only_summary.to_excel(
        writer,
        sheet_name="better_only_summary",
        index=False
    )

    median_error_df.to_excel(
        writer,
        sheet_name="median_error_table",
        index=False
    )

    observed_range_df.to_excel(
        writer,
        sheet_name="observed_error_ranges",
        index=False
    )

print(f"Saved numerical analysis workbook to: {NUMERICAL_OUTPUT_PATH}")

Saved numerical analysis workbook to: c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\numerical_qos_relative_analysis.xlsx


In [ ]:
TARGET_ALGORITHMS = ["IT-QOS-OFA", "IT-SIA-QOS-OFA"]

worse_cases = (
    numerical_qos_relative[
        (numerical_qos_relative["algorithm"].isin(TARGET_ALGORITHMS)) &
        (numerical_qos_relative["qos_relative_retention_percent"] < 100)
    ]
    .copy()
)

# Degradation magnitude:
# If retention = 64.08, degradation = 100 - 64.08 = 35.92
worse_cases["degradation_when_worse_percent"] = (
    100.0 - worse_cases["qos_relative_retention_percent"]
)

# Function-level details
worse_by_function = worse_cases[
    [
        "cec_func_num",
        "objective_name",
        "algorithm",
        "qos_relative_retention_percent",
        "degradation_when_worse_percent",
    ]
].sort_values(["algorithm", "cec_func_num"]).reset_index(drop=True)

# Summary table for both algorithms
worse_summary = (
    worse_by_function
    .groupby("algorithm", as_index=False)
    .agg(
        worse_than_qos_functions=("cec_func_num", "count"),
        mean_degradation_when_worse_percent=(
            "degradation_when_worse_percent", "mean"
        ),
        median_degradation_when_worse_percent=(
            "degradation_when_worse_percent", "median"
        ),
        max_degradation_percent=(
            "degradation_when_worse_percent", "max"
        ),
    )
)

display(worse_summary)
display(worse_by_function)

SHEET_NAME = "degradation_when_worse"

with pd.ExcelWriter(
    NUMERICAL_OUTPUT_PATH,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    worse_summary.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        startrow=0
    )
    
    worse_by_function.to_excel(
        writer,
        sheet_name=SHEET_NAME,
        index=False,
        startrow=4
    )

print(f"Added sheet '{SHEET_NAME}' to:")
print(NUMERICAL_OUTPUT_PATH)

,algorithm,worse_than_qos_functions,mean_degradation_when_worse_percent,median_degradation_when_worse_percent,max_degradation_percent
0,IT-QOS-OFA,11,20.087139,16.693652,89.372491
1,IT-SIA-QOS-OFA,7,29.389440,35.919793,49.139686


,cec_func_num,objective_name,algorithm,qos_relative_retention_percent,degradation_when_worse_percent
0,3,CEC Zakharov,IT-QOS-OFA,94.121526,5.878474
1,5,CEC Rastrigin,IT-QOS-OFA,75.290081,24.709919
2,6,CEC Expanded Scaffer's F6,IT-QOS-OFA,83.306348,16.693652
3,9,CEC Levy,IT-QOS-OFA,99.579553,0.420447
4,11,CEC Hybrid Function 1,IT-QOS-OFA,95.092541,4.907459
5,16,CEC Hybrid Function 6,IT-QOS-OFA,75.306506,24.693494
6,20,CEC Hybrid Function 10,IT-QOS-OFA,88.305106,11.694894
7,22,CEC Composition Function 2,IT-QOS-OFA,10.627509,89.372491
8,24,CEC Composition Function 4,IT-QOS-OFA,83.238960,16.761040
9,27,CEC Composition Function 7,IT-QOS-OFA,79.476240,20.523760


Added sheet 'degradation_when_worse' to:
c:\Users\kvrgi\Desktop\Thesis---Sandro\Thesis_code\analysis_exports\numerical_qos_relative_analysis.xlsx


In [ ]:
# Defining which functions should be removed for each algorithm for sensitivity analysis
functions_to_remove = {
    "IT-QOS-OFA": [9, 22],
    "IT-SIA-QOS-OFA": [6, 16],
}

# Create sensitivity-analysis dataframe
sensitivity_df = numerical_qos_relative.copy()

# Remove algorithm-specific functions
for algorithm, funcs in functions_to_remove.items():
    sensitivity_df = sensitivity_df[
        ~(
            (sensitivity_df["algorithm"] == algorithm)
            & (sensitivity_df["cec_func_num"].isin(funcs))
        )
    ]

# Compute sensitivity summary
sensitivity_summary = (
    sensitivity_df
    .groupby("algorithm", as_index=False)
    .agg(
        mean_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "mean"
        ),
        median_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "median"
        ),
        functions_better_than_qos=(
            "direction_vs_qos", lambda x: int((x == "better_than_QOS").sum())
        ),
        functions_worse_than_qos=(
            "direction_vs_qos", lambda x: int((x == "worse_than_QOS").sum())
        ),
        functions_same_as_qos=(
            "direction_vs_qos", lambda x: int((x == "same_as_QOS").sum())
        ),
        functions_remaining=(
            "cec_func_num", "count"
        ),
    )
)

display(sensitivity_summary)

,algorithm,mean_qos_relative_retention_percent,median_qos_relative_retention_percent,functions_better_than_qos,functions_worse_than_qos,functions_same_as_qos,functions_remaining
0,IT-QOS-OFA,85.426045,83.306348,0,9,0,9
1,IT-SIA-QOS-OFA,83.078006,94.181654,3,6,0,9


In [20]:
# Define which functions should be removed for each algorithm
functions_to_remove = {
    "IT-QOS-OFA": [22],
    "IT-SIA-QOS-OFA": [6],
}

# Create sensitivity-analysis dataframe
sensitivity_df = numerical_qos_relative.copy()

# Remove algorithm-specific functions
for algorithm, funcs in functions_to_remove.items():
    sensitivity_df = sensitivity_df[
        ~(
            (sensitivity_df["algorithm"] == algorithm)
            & (sensitivity_df["cec_func_num"].isin(funcs))
        )
    ]

# Compute sensitivity summary
sensitivity_summary = (
    sensitivity_df
    .groupby("algorithm", as_index=False)
    .agg(
        mean_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "mean"
        ),
        median_qos_relative_retention_percent=(
            "qos_relative_retention_percent", "median"
        ),
        functions_better_than_qos=(
            "direction_vs_qos", lambda x: int((x == "better_than_QOS").sum())
        ),
        functions_worse_than_qos=(
            "direction_vs_qos", lambda x: int((x == "worse_than_QOS").sum())
        ),
        functions_same_as_qos=(
            "direction_vs_qos", lambda x: int((x == "same_as_QOS").sum())
        ),
        functions_remaining=(
            "cec_func_num", "count"
        ),
    )
)

display(sensitivity_summary)

,algorithm,mean_qos_relative_retention_percent,median_qos_relative_retention_percent,functions_better_than_qos,functions_worse_than_qos,functions_same_as_qos,functions_remaining
0,IT-QOS-OFA,86.841396,85.805727,0,10,0,10
1,IT-SIA-QOS-OFA,79.856237,85.352770,3,7,0,10


In [ ]:
CONVERGENCE_FILES = {
    "QOS-OFA": Path("convergence_QOS_OFA_all_usable_cec.xlsx"),
    "IT-QOS-OFA": Path("convergence_IT_QOS_OFA_all_usable_cec.xlsx"),
    "IT-SIA-QOS-OFA": Path("convergence_IT_SIA_QOS_OFA_all_usable_cec.xlsx"),
}

SEARCH_RANGE_SHEET = "search_range_by_function"

search_range_parts = []

for algorithm, path in CONVERGENCE_FILES.items():
    print(f"\nChecking {algorithm}: {path}")

    if not path.exists():
        raise FileNotFoundError(f"Could not find convergence workbook: {path}")

    xls = pd.ExcelFile(path)

    print("Sheets:", xls.sheet_names)

    if SEARCH_RANGE_SHEET not in xls.sheet_names:
        raise ValueError(
            f"Sheet '{SEARCH_RANGE_SHEET}' not found in {path}.\n"
            f"Available sheets are: {xls.sheet_names}\n\n"
            "This means the convergence workbook was probably created before "
            "we added the search-range export. Rerun the patched convergence cell "
            "for this algorithm notebook."
        )

    df = pd.read_excel(path, sheet_name=SEARCH_RANGE_SHEET)

    df["algorithm"] = algorithm

    required_cols = [
        "algorithm",
        "cec_func_num",
        "objective_name",
        "search_min_error_algorithm",
        "search_max_error_algorithm",
        "search_range_algorithm",
    ]

    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Missing columns in {path}, sheet '{SEARCH_RANGE_SHEET}': {missing_cols}"
        )

    df = df[required_cols].copy()

    search_range_parts.append(df)

search_range_algorithm_df = pd.concat(search_range_parts, ignore_index=True)

search_range_algorithm_df["cec_func_num"] = (
    search_range_algorithm_df["cec_func_num"].astype(int)
)

display(search_range_algorithm_df)


Checking QOS-OFA: convergence_QOS_OFA_all_usable_cec.xlsx
Sheets: ['convergence_all', 'per_run_all', 'config', 'search_range_by_function']

Checking IT-QOS-OFA: convergence_IT_QOS_OFA_all_usable_cec.xlsx
Sheets: ['convergence_all', 'per_run_all', 'config', 'search_range_by_function']

Checking IT-SIA-QOS-OFA: convergence_IT_SIA_QOS_OFA_all_usable_cec.xlsx
Sheets: ['convergence_all', 'per_run_all', 'config', 'search_range_by_function']


,algorithm,cec_func_num,objective_name,search_min_error_algorithm,search_max_error_algorithm,search_range_algorithm
0,QOS-OFA,3,CEC Zakharov,0.050842,1.244990e+15,1.244990e+15
1,QOS-OFA,5,CEC Rastrigin,24.906006,3.668622e+05,3.668373e+05
2,QOS-OFA,6,CEC Expanded Scaffer's F6,11.855591,1.743707e+01,5.581482e+00
3,QOS-OFA,9,CEC Levy,0.007080,1.281403e+05,1.281403e+05
4,QOS-OFA,11,CEC Hybrid Function 1,16.136353,1.823792e+11,1.823792e+11
5,QOS-OFA,16,CEC Hybrid Function 6,4.778809,1.796274e+05,1.796226e+05
6,QOS-OFA,20,CEC Hybrid Function 10,62.182373,4.564127e+03,4.501945e+03
7,QOS-OFA,22,CEC Composition Function 2,100.273682,1.715693e+04,1.705665e+04
8,QOS-OFA,24,CEC Composition Function 4,374.608154,6.532807e+03,6.158198e+03
9,QOS-OFA,27,CEC Composition Function 7,450.017090,1.834165e+04,1.789164e+04


In [ ]:
function_names_df = (
    search_range_algorithm_df
    .sort_values(["cec_func_num", "algorithm"])
    .drop_duplicates(subset=["cec_func_num"])
    [["cec_func_num", "objective_name"]]
)

search_range_by_function = (
    search_range_algorithm_df
    .groupby("cec_func_num", as_index=False)
    .agg(
        E_f_S_min=("search_min_error_algorithm", "min"),
        E_f_S_max=("search_max_error_algorithm", "max"),
    )
    .merge(function_names_df, on="cec_func_num", how="left")
)

search_range_by_function["S_f"] = (
    search_range_by_function["E_f_S_max"]
    - search_range_by_function["E_f_S_min"]
)

search_range_by_function = search_range_by_function[
    [
        "cec_func_num",
        "objective_name",
        "E_f_S_min",
        "E_f_S_max",
        "S_f",
    ]
].sort_values("cec_func_num").reset_index(drop=True)

if (search_range_by_function["S_f"] <= 0).any():
    display(search_range_by_function[search_range_by_function["S_f"] <= 0])
    raise ValueError("At least one function has S_f <= 0.")

display(search_range_by_function)

,cec_func_num,objective_name,E_f_S_min,E_f_S_max,S_f
0,3,CEC Zakharov,0.002014,1.244990e+15,1.244990e+15
1,5,CEC Rastrigin,24.906006,3.993540e+05,3.993291e+05
2,6,CEC Expanded Scaffer's F6,10.450012,1.743707e+01,6.987061e+00
3,9,CEC Levy,0.007080,1.328786e+05,1.328786e+05
4,11,CEC Hybrid Function 1,16.136353,2.064348e+11,2.064348e+11
5,16,CEC Hybrid Function 6,4.778809,2.254127e+05,2.254079e+05
6,20,CEC Hybrid Function 10,62.182373,4.564127e+03,4.501945e+03
7,22,CEC Composition Function 2,100.000977,1.715693e+04,1.705693e+04
8,24,CEC Composition Function 4,374.608154,6.594118e+03,6.219510e+03
9,27,CEC Composition Function 7,450.017090,1.834165e+04,1.789164e+04


In [ ]:
search_normalized_qos_relative = numerical_qos_relative.copy()

search_normalized_qos_relative = search_normalized_qos_relative.merge(
    search_range_by_function[
        ["cec_func_num", "E_f_S_min", "E_f_S_max", "S_f"]
    ],
    on="cec_func_num",
    how="left"
)

missing_s = search_normalized_qos_relative[
    search_normalized_qos_relative["S_f"].isna()
]

if not missing_s.empty:
    display(missing_s)
    raise ValueError("Some rows are missing S_f values.")

search_normalized_qos_relative["error_difference_vs_qos"] = (
    search_normalized_qos_relative["median_error"]
    - search_normalized_qos_relative["qos_median_error"]
)

search_normalized_qos_relative["signed_search_normalized_difference_percent"] = (
    100.0
    * search_normalized_qos_relative["error_difference_vs_qos"]
    / search_normalized_qos_relative["S_f"]
)

search_normalized_qos_relative["qos_relative_retention_search_percent"] = (
    100.0
    - search_normalized_qos_relative["signed_search_normalized_difference_percent"]
)

search_normalized_qos_relative["closeness_to_qos_search_percent"] = (
    100.0
    - search_normalized_qos_relative[
        "signed_search_normalized_difference_percent"
    ].abs()
)

search_normalized_qos_relative["direction_vs_qos"] = np.select(
    [
        search_normalized_qos_relative["error_difference_vs_qos"] < 0,
        search_normalized_qos_relative["error_difference_vs_qos"] > 0,
    ],
    [
        "better_than_QOS",
        "worse_than_QOS",
    ],
    default="same_as_QOS"
)

display(search_normalized_qos_relative)

,cec_func_num,objective_name,algorithm,qos_median_error,median_error,error_difference_vs_qos,observed_min_error,observed_max_error,observed_error_range,signed_normalized_difference_percent,...,mean_error,std_error,best_error,worst_error,E_f_S_min,E_f_S_max,S_f,signed_search_normalized_difference_percent,qos_relative_retention_search_percent,closeness_to_qos_search_percent
0,3,CEC Zakharov,IT-QOS-OFA,0.283447,1.196119,0.912672,0.002014,15.527679,15.525665,5.878474,...,2.436207,3.641462,0.229113,15.527679,0.002014,1.244990e+15,1.244990e+15,7.330760e-14,100.000000,100.000000
1,5,CEC Rastrigin,IT-QOS-OFA,48.757935,104.831609,56.073674,24.906006,251.833801,226.927795,24.709919,...,106.813636,31.325394,49.345574,163.626186,24.906006,3.993540e+05,3.993291e+05,1.404197e-02,99.985958,99.985958
2,6,CEC Expanded Scaffer's F6,IT-QOS-OFA,13.134979,13.766882,0.631903,10.450012,14.235301,3.785289,16.693652,...,13.763935,0.245770,13.105189,14.235301,10.450012,1.743707e+01,6.987061e+00,9.043903e+00,90.956097,90.956097
3,9,CEC Levy,IT-QOS-OFA,0.099609,2.006804,1.907194,0.007080,453.618286,453.611206,0.420447,...,3.946485,7.841389,0.832761,43.723286,0.007080,1.328786e+05,1.328786e+05,1.435291e-03,99.998565,99.998565
4,11,CEC Hybrid Function 1,IT-QOS-OFA,22.670593,66.084845,43.414252,16.136353,900.794749,884.658397,4.907459,...,143.447544,197.531833,29.154732,900.794749,16.136353,2.064348e+11,2.064348e+11,2.103049e-08,100.000000,100.000000
5,16,CEC Hybrid Function 6,IT-QOS-OFA,263.244873,621.188127,357.943254,4.785767,1454.330566,1449.544800,24.693494,...,620.418791,201.747811,153.213904,1089.811619,4.778809,2.254127e+05,2.254079e+05,1.587980e-01,99.841202,99.841202
6,20,CEC Hybrid Function 10,IT-QOS-OFA,185.239868,227.994241,42.754373,62.182373,427.763916,365.581543,11.694894,...,218.867115,68.075043,107.158966,359.774272,62.182373,4.564127e+03,4.501945e+03,9.496867e-01,99.050313,99.050313
7,22,CEC Composition Function 2,IT-QOS-OFA,100.508911,116.328370,15.819459,100.000977,117.701567,17.700590,89.372491,...,116.078350,0.900819,114.545842,117.701567,100.000977,1.715693e+04,1.705693e+04,9.274507e-02,99.907255,99.907255
8,24,CEC Composition Function 4,IT-QOS-OFA,412.235107,444.867985,32.632878,374.608398,569.303223,194.694824,16.761040,...,444.149137,19.542250,408.055394,484.924707,374.608154,6.594118e+03,6.219510e+03,5.246857e-01,99.475314,99.475314
9,27,CEC Composition Function 7,IT-QOS-OFA,455.984131,464.944285,8.960155,450.041748,493.699219,43.657471,20.523760,...,464.568323,7.890212,452.203492,479.417146,450.017090,1.834165e+04,1.789164e+04,5.008013e-02,99.949920,99.949920


# Final summary

In [ ]:
# Final summary of QOS-relative performance normalized by search range
search_normalized_summary = (
    search_normalized_qos_relative
    .groupby("algorithm", as_index=False)
    .agg(
        mean_P_Af_S=(
            "qos_relative_retention_search_percent", "mean"
        ),
        median_P_Af_S=(
            "qos_relative_retention_search_percent", "median"
        ),
        min_P_Af_S=(
            "qos_relative_retention_search_percent", "min"
        ),
        max_P_Af_S=(
            "qos_relative_retention_search_percent", "max"
        ),
        functions_better_than_qos=(
            "direction_vs_qos", lambda x: int((x == "better_than_QOS").sum())
        ),
        functions_worse_than_qos=(
            "direction_vs_qos", lambda x: int((x == "worse_than_QOS").sum())
        ),
        functions_same_as_qos=(
            "direction_vs_qos", lambda x: int((x == "same_as_QOS").sum())
        ),
    )
)

display(search_normalized_summary)

,algorithm,mean_P_Af_S,median_P_Af_S,min_P_Af_S,max_P_Af_S,functions_better_than_qos,functions_worse_than_qos,functions_same_as_qos
0,IT-QOS-OFA,99.014493,99.949920,90.956097,100.000000,0,11,0
1,IT-SIA-QOS-OFA,101.404295,99.980138,98.093616,119.267357,4,7,0


: 